# 365 Probabilidades - Dia #149
## Qual a probabilidade de cada centímetro a mais valer dinheiro?

**Tipo:** Trabalho · Carreira
**Data de publicação:** 2026-11-09
**Ferramenta:** Python
**Decisão analisada:** O que pesa no meu salário além do que eu faço?
**Hashtag:** #365Probabilidades #Dia149

---

### 📖 A História

Ninguém escolhe a própria altura. Ela chega pronta, entra na sala antes da gente, aparece na
foto do crachá e na primeira impressão da entrevista. E quase ninguém imagina que ela possa
aparecer também no contracheque.

A ideia de que gente alta ganha mais circula há décadas, quase sempre como piada. Fui ver se
ela se sustenta nos dados, se o efeito é causa ou só coincidência, e, principalmente, quanto
ele vale no Brasil, com a altura e o salário das mesmas pessoas.

---

### 📚 O Conceito: o prêmio da altura

Em 2004, Timothy Judge e Daniel Cable juntaram quatro grandes estudos que acompanharam milhares
de pessoas da infância à vida adulta, três nos Estados Unidos e um no Reino Unido. Descontados
sexo, peso e idade, cada polegada (2,54 cm) de altura valia cerca de US$ 789 a mais por ano, e
o efeito não diminuía com a idade.

Associação não é causa. Gente alta pode ter crescido com mais comida, mais saúde e mais escola.
Em 2016, Jessica Tyrrell e colegas usaram um atalho genético para separar uma coisa da outra.
Os genes que influenciam a altura são sorteados na concepção, antes de qualquer vantagem de
família, e funcionam como um experimento natural (a randomização mendeliana). Com 119.669
pessoas do UK Biobank, ser 6,3 cm mais alto **causou** cerca de £ 1.130 a mais de renda
familiar por ano, e 12% mais chance de estar numa ocupação qualificada.

Anne Case e Christina Paxson (2008) mostraram o outro lado. Crianças mais altas já vão melhor
em testes cognitivos aos 3 anos, e boa parte do prêmio da altura adulta se explica por isso.
A altura funciona em parte como um registro da infância que a pessoa teve. A revisão de
Thompson, Portrait e Schoonmade (2023), com 42 estudos, encontrou o prêmio maior nos homens e
maior na América Latina e na Ásia do que nos Estados Unidos.

---

### 🧮 O Modelo

**Fontes:**
- Judge, T. A. & Cable, D. M. (2004). The effect of physical height on workplace success and
  income. *Journal of Applied Psychology*, 89(3), 428-441. Quatro estudos longitudinais (EUA e
  Reino Unido). US$ 789 por polegada por ano. N de cada estudo e parte de meta-análise não
  conferidos no texto completo.
- Tyrrell, J. et al. (2016). Height, body mass index, and socioeconomic status: mendelian
  randomisation study in UK Biobank. *BMJ*, 352, i582. N = 119.669. +6,3 cm causa +£ 1.130 de
  renda familiar anual (IC 95% £ 680 a £ 1.580).
- Case, A. & Paxson, C. (2008). Stature and status. *Journal of Political Economy*, 116(3),
  499-532. Quatro bases dos EUA e Reino Unido; o prêmio se explica em grande parte pela
  capacidade cognitiva medida na infância.
- Thompson, K., Portrait, F. & Schoonmade, L. (2023). The height premium. *Economics and Human
  Biology*, 50, 101273. 42 estudos; prêmio maior em homens e na América Latina.
- **Dado próprio:** IBGE, Pesquisa Nacional de Saúde 2019, microdados (morador selecionado de
  15 anos ou mais). Altura informada por 75.635 pessoas e medida por balança e estadiômetro numa
  subamostra. Rendimento bruto mensal do trabalho principal e horas semanais. Recorte de 25 a 59
  anos, ocupados, com rendimento em dinheiro: 34.549 pessoas (19.908 homens e 14.641 mulheres).
  Arquivo reduzido em `dados149/dia-149-pns2019-altura-renda.csv`, gerado a partir de
  `PNS_2019_20220525.zip` e do input oficial.

**Desenho e família do teste:** regressão linear do logaritmo da renda na altura (em 10 cm),
ponderada pelo peso amostral do morador selecionado, com erro-padrão agrupado por UPA (o
conglomerado da amostra), separada por sexo. Três modelos: só idade; idade mais instrução, cor,
UF e tipo de área; e o mesmo com o rendimento por hora. A probabilidade final vem de um Monte
Carlo de pares de pessoas iguais em tudo menos na altura, sorteando os resíduos do modelo e a
incerteza do coeficiente.

**Sem ×0,80.** São coeficientes de regressão, não proporções de pesquisa autorrelatada. A altura
informada é autorrelato, e a camada 4 mede quanto ela se afasta da altura medida.

**Cordas declaradas:** #053 (a sorte na carreira), #120 (chegar mais longe que os pais) e #086
(o ano em que você começou). Objetos distintos.


In [ ]:
import os, zipfile, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

SEMENTE = 149
rng = np.random.default_rng(SEMENTE)
SORTEIOS = 200_000

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
RODAPE = ('Fonte: IBGE, PNS 2019, microdados · Judge & Cable 2004 · Tyrrell et al. 2016 · Case & Paxson 2008 · '
          'cálculo próprio | #365Probabilidades')

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def mil(n):
    return f"{int(round(n)):,}".replace(",", ".")

def pct(b):
    return (np.exp(b) - 1) * 100

print("✅ Bibliotecas carregadas · semente", SEMENTE)


In [ ]:
# --- DADOS ---
ARQ = 'dados149/dia-149-pns2019-altura-renda.csv'

if not os.path.exists(ARQ):
    # gera o arquivo reduzido a partir dos microdados oficiais (uma vez só)
    VARS = {'UPA_PNS': 'upa', 'V0024': 'estrato', 'V0001': 'uf', 'V0031': 'tipo_area', 'C006': 'sexo', 'C008': 'idade',
            'C009': 'cor', 'VDD004A': 'instrucao', 'E001': 'trabalhou', 'E01601': 'recebe_dinheiro',
            'E01602': 'renda_mensal', 'E017': 'horas_semana', 'P00402': 'sabe_altura', 'P00403': 'altura_informada',
            'W00203': 'altura_medida', 'V00291': 'peso_morador', 'V00301': 'peso_antropometria'}
    with zipfile.ZipFile('Dicionario_e_input_20220530.zip') as z:
        txt = z.read('input_PNS_2019.txt').decode('latin-1')
    pos = {m.group(2): (int(m.group(1)) - 1, int(m.group(3)))
           for m in re.finditer(r'@(\d+)\s+(\S+)\s+\$?(\d+)\.', txt)}
    with zipfile.ZipFile('PNS_2019_20220525.zip') as z, z.open('PNS_2019.txt') as f:
        bruto = pd.read_fwf(f, colspecs=[(pos[v][0], pos[v][0] + pos[v][1]) for v in VARS],
                            names=list(VARS.values()), dtype=str, encoding='latin-1')
    bruto = bruto[bruto['peso_morador'].notna()]
    os.makedirs('dados149', exist_ok=True)
    bruto.to_csv(ARQ, index=False)

pns = pd.read_csv(ARQ, dtype={'upa': str, 'estrato': str, 'uf': str, 'tipo_area': str, 'sexo': str, 'cor': str,
                              'instrucao': str, 'trabalhou': str, 'recebe_dinheiro': str, 'sabe_altura': str})
pns['altura_medida'] = pd.to_numeric(pns['altura_medida'], errors='coerce')
if pns['altura_medida'].max() > 300:
    pns['altura_medida'] = pns['altura_medida'] / 10      # o input traz uma casa decimal implícita

amostra = pns[(pns['sabe_altura'] == '1') & pns['altura_informada'].between(130, 210) & pns['idade'].between(25, 59)
              & (pns['trabalhou'] == '1') & (pns['recebe_dinheiro'] == '1') & (pns['renda_mensal'] > 0)
              & (pns['renda_mensal'] < 99_999_999) & (pns['horas_semana'] > 0)].copy()
amostra['sexo_nome'] = np.where(amostra['sexo'] == '2', 'Mulheres', 'Homens')
amostra['alt10'] = amostra['altura_informada'] / 10
amostra['lrenda'] = np.log(amostra['renda_mensal'])
amostra['lhora'] = np.log(amostra['renda_mensal'] / (amostra['horas_semana'] * 4.33))
amostra['regiao'] = amostra['uf'].str[0]
for c in ('instrucao', 'cor', 'tipo_area'):
    amostra[c] = amostra[c].fillna('sem')
amostra['superior'] = (amostra['instrucao'] == '7').astype(float)

# Literatura
JUDGE_CABLE_POL = 789          # US$ por polegada por ano
TYRRELL = {'cm': 6.3, 'libras': 1130, 'ic': (680, 1580), 'n': 119_669}

print("=" * 72)
print("  DADOS · PNS 2019, morador selecionado")
print("=" * 72)
print(f"  Moradores selecionados no arquivo: {len(pns):,}".replace(",", "."))
print(f"  Sabem a altura e informaram: {(pns['sabe_altura'] == '1').sum():,}".replace(",", "."))
print(f"  Altura medida (subamostra de antropometria): {pns['altura_medida'].notna().sum():,}".replace(",", "."))
print(f"  Recorte 25 a 59 anos, ocupados e com rendimento: {mil(len(amostra))}")
for s, g in amostra.groupby('sexo_nome'):
    alt = np.average(g['altura_informada'], weights=g['peso_morador'])
    print(f"    {s:<9} {mil(len(g)):>6} pessoas · altura média {vir(alt)} cm")
print(f"\n  Judge & Cable 2004: US$ {JUDGE_CABLE_POL} por polegada por ano (cerca de US$ {round(JUDGE_CABLE_POL / 2.54)} por cm)")
print(f"  Tyrrell et al. 2016 (N = {mil(TYRRELL['n'])}): +{vir(TYRRELL['cm'])} cm causa +£ {mil(TYRRELL['libras'])} por ano")
print("=" * 72)


In [ ]:
# --- O MODELO ---
FORMULAS = {
    'só idade': 'lrenda ~ alt10 + idade + I(idade**2)',
    'mesma escola, cor e lugar': 'lrenda ~ alt10 + idade + I(idade**2) + C(instrucao) + C(cor) + C(uf) + C(tipo_area)',
    'por hora trabalhada': 'lhora ~ alt10 + idade + I(idade**2) + C(instrucao) + C(cor) + C(uf) + C(tipo_area)',
}
RES = {}
for s, g in amostra.groupby('sexo_nome'):
    for nome, f in FORMULAS.items():
        m = smf.wls(f, data=g, weights=g['peso_morador']).fit(cov_type='cluster', cov_kwds={'groups': g['upa']})
        b, se = m.params['alt10'], m.bse['alt10']
        sd = np.sqrt(np.average(m.resid ** 2, weights=g['peso_morador']))
        RES[(s, nome)] = {'b': b, 'se': se, 'lo': b - 1.96 * se, 'hi': b + 1.96 * se, 'sd': sd, 'resid': m.resid.to_numpy(),
                          'w': g['peso_morador'].to_numpy(), 'n': int(m.nobs)}

# CAMADA 1 · O Brasil, sem descontar nada além da idade
print("=" * 72)
print("  CAMADA 1 · QUANTO VALEM 10 CM NO BRASIL (rendimento mensal)")
print("=" * 72)
for s in ('Homens', 'Mulheres'):
    for nome in FORMULAS:
        r = RES[(s, nome)]
        print(f"  {s:<9} {nome:<28} {vir(pct(r['b'])):>6}%  IC 95% [{vir(pct(r['lo']))}; {vir(pct(r['hi']))}]  n = {mil(r['n'])}")
    print()

# CAMADA 2 · O que a escola explica
print("=" * 72)
print("  CAMADA 2 · O QUE A ESCOLA E O LUGAR EXPLICAM")
print("=" * 72)
for s in ('Homens', 'Mulheres'):
    b0, b1 = RES[(s, 'só idade')]['b'], RES[(s, 'mesma escola, cor e lugar')]['b']
    print(f"  {s:<9} o prêmio cai de {vir(pct(b0))}% para {vir(pct(b1))}% por 10 cm · a escola, a cor e o lugar explicam "
          f"{vir((1 - b1 / b0) * 100, 0)}% do coeficiente")
faixas = [0, 155, 160, 165, 170, 175, 180, 250]
rot_faixas = ['até 155', '156 a 160', '161 a 165', '166 a 170', '171 a 175', '176 a 180', 'acima de 180']
amostra['faixa'] = pd.cut(amostra['altura_informada'], faixas, labels=rot_faixas)
TAB = amostra.groupby(['sexo_nome', 'faixa'], observed=True).apply(
    lambda x: pd.Series({'pessoas': len(x), 'renda_mediana': x['renda_mensal'].median(),
                         'superior': np.average(x['superior'], weights=x['peso_morador'])}))
print("\n  Renda mediana e diploma de ensino superior por faixa de altura")
for (s, f), row in TAB.iterrows():
    print(f"    {s:<9} {f:<13} {int(row['pessoas']):>5} pessoas · mediana R$ {row['renda_mediana']:,.0f} · superior {vir(row['superior'] * 100, 0)}%".replace(",", "."))

# CAMADA 3 · A probabilidade, em pares de pessoas
def prob_par(r, cm=10, sorteios=SORTEIOS):
    b = rng.normal(r['b'], r['se'], sorteios) * cm / 10
    p = r['w'] / r['w'].sum()
    e1 = rng.choice(r['resid'], sorteios, p=p)
    e2 = rng.choice(r['resid'], sorteios, p=p)
    return (b + e1 - e2 > 0).mean()

PROB = {}
for s in ('Homens', 'Mulheres'):
    PROB[(s, 'iguais, 10 cm')] = prob_par(RES[(s, 'mesma escola, cor e lugar')], 10)
    PROB[(s, 'iguais, 20 cm')] = prob_par(RES[(s, 'mesma escola, cor e lugar')], 20)
    PROB[(s, 'só a idade igual, 10 cm')] = prob_par(RES[(s, 'só idade')], 10)
print("\n" + "=" * 72)
print("  CAMADA 3 · DUAS PESSOAS, UMA MAIS ALTA (Monte Carlo com " + f"{SORTEIOS:,}".replace(",", ".") + " pares)")
print("=" * 72)
for s in ('Homens', 'Mulheres'):
    print(f"  {s}")
    for k in ('iguais, 10 cm', 'iguais, 20 cm', 'só a idade igual, 10 cm'):
        print(f"    {k:<26} a mais alta ganha mais em {vir(PROB[(s, k)] * 100)}% dos pares (sem efeito da altura = 50%)")

# CAMADA 4 · A altura que a gente informa
med = pns[pns['altura_medida'].between(120, 220) & (pns['sabe_altura'] == '1') & pns['altura_informada'].between(130, 210)].copy()
med['dif'] = med['altura_informada'] - med['altura_medida']
print("\n" + "=" * 72)
print("  CAMADA 4 · A ALTURA INFORMADA CONTRA A MEDIDA")
print("=" * 72)
print(f"  Pessoas com as duas medidas: {len(med):,}".replace(",", "."))
for sx, nome in (('1', 'Homens'), ('2', 'Mulheres')):
    g = med[med['sexo'] == sx]
    print(f"    {nome:<9} informam {vir(g['dif'].mean())} cm a mais, em média (desvio {vir(g['dif'].std())} cm)")
print(f"  Correlação entre informada e medida: {vir(med[['altura_informada', 'altura_medida']].corr().iloc[0, 1], 2)}")
sub = amostra[amostra['altura_medida'].between(120, 220)].copy()
sub['altm10'] = sub['altura_medida'] / 10
CHECK = {}
for s, g in sub.groupby('sexo_nome'):
    for var in ('alt10', 'altm10'):
        m = smf.wls(FORMULAS['mesma escola, cor e lugar'].replace('alt10', var), data=g,
                    weights=g['peso_antropometria']).fit(cov_type='cluster', cov_kwds={'groups': g['upa']})
        CHECK[(s, var)] = (m.params[var], m.conf_int().loc[var].to_numpy(), int(m.nobs))
for s in ('Homens', 'Mulheres'):
    (bi, ci, n), (bm, cm_, _) = CHECK[(s, 'alt10')], CHECK[(s, 'altm10')]
    print(f"    {s:<9} n = {mil(n)} · informada {vir(pct(bi))}% [{vir(pct(ci[0]))}; {vir(pct(ci[1]))}] · "
          f"medida {vir(pct(bm))}% [{vir(pct(cm_[0]))}; {vir(pct(cm_[1]))}]")
print("  Com a altura medida, o prêmio fica parecido. Os intervalos são largos porque a subamostra é pequena.")
print("=" * 72)


In [ ]:
# --- VISUALIZAÇÃO ---

# GRÁFICO 1 · A renda e o diploma sobem com a altura
fig1, (ax1a, ax1b) = plt.subplots(1, 2, figsize=(14, 7), sharey=False)
for ax, s, cor in ((ax1a, 'Homens', CINZA), (ax1b, 'Mulheres', VERMELHO)):
    t = TAB.loc[s]
    t = t[t['pessoas'] >= 100]
    x = np.arange(len(t))
    ax.bar(x, t['renda_mediana'], color=cor, alpha=0.85, width=0.6)
    for i, (f, row) in enumerate(t.iterrows()):
        ax.text(i, row['renda_mediana'] + 40, f"R$ {row['renda_mediana']:,.0f}".replace(",", "."), ha='center', fontsize=9.5)
        ax.text(i, 120, f"{vir(row['superior'] * 100, 0)}%\nsuperior", ha='center', fontsize=8.5, color='white')
    ax.set_xticks(x)
    ax.set_xticklabels([f"{f}\ncm" for f in t.index], fontsize=9)
    ax.set_title(s, fontsize=13)
    ax.set_ylim(0, 2400)
ax1a.set_ylabel('Rendimento mensal mediano do trabalho (R$)', fontsize=11)
fig1.suptitle('Quanto mais alto, maior a renda e maior a chance de ter diploma\nPNS 2019, 25 a 59 anos, ocupados · '
              'faixas com 100 pessoas ou mais', fontsize=14)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 0.95))
plt.savefig('dia-149-grafico-01-a-escada-da-altura.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# GRÁFICO 2 · O que sobra depois de descontar escola, cor e lugar
fig2, ax2 = plt.subplots(figsize=(12, 7))
nomes = list(FORMULAS)
x = np.arange(len(nomes))
for j, (s, cor) in enumerate((('Homens', CINZA), ('Mulheres', VERMELHO))):
    v = [pct(RES[(s, n)]['b']) for n in nomes]
    lo = [pct(RES[(s, n)]['b']) - pct(RES[(s, n)]['lo']) for n in nomes]
    hi = [pct(RES[(s, n)]['hi']) - pct(RES[(s, n)]['b']) for n in nomes]
    ax2.bar(x + (j - 0.5) * 0.38, v, width=0.36, color=cor, alpha=0.9, label=s)
    ax2.errorbar(x + (j - 0.5) * 0.38, v, yerr=[lo, hi], fmt='none', ecolor='#333', capsize=5)
    for i, val in enumerate(v):
        ax2.text(i + (j - 0.5) * 0.38, val + hi[i] + 0.8, f"{vir(val)}%", ha='center', fontsize=11, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(['comparando só pela idade', 'mesma escola, cor e lugar', 'mesma escola, por hora trabalhada'], fontsize=11)
ax2.set_ylabel('Renda a mais para cada 10 cm de altura (%) · IC 95%', fontsize=11)
ax2.set_title('Dois terços do prêmio da altura são escola e origem. O resto continua lá\n'
              'regressão ponderada, erro-padrão por conglomerado', fontsize=13, pad=14)
ax2.legend(frameon=False, fontsize=11)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-149-grafico-02-o-que-sobra.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# GRÁFICO 3 · Os pares
fig3, ax3 = plt.subplots(figsize=(12, 7))
casos = ['iguais, 10 cm', 'iguais, 20 cm', 'só a idade igual, 10 cm']
rot = ['iguais em tudo,\n10 cm de diferença', 'iguais em tudo,\n20 cm de diferença', 'só a idade igual,\n10 cm de diferença']
x = np.arange(len(casos))
for j, (s, cor) in enumerate((('Homens', CINZA), ('Mulheres', VERMELHO))):
    v = [PROB[(s, c)] * 100 for c in casos]
    ax3.bar(x + (j - 0.5) * 0.38, v, width=0.36, color=cor, alpha=0.9, label=s)
    for i, val in enumerate(v):
        ax3.text(i + (j - 0.5) * 0.38, val + 0.4, f"{vir(val, 0)}%", ha='center', fontsize=12, fontweight='bold')
ax3.axhline(50, color=DOURADO, linestyle='--', linewidth=2)
ax3.text(-0.45, 50.4, 'se a altura não importasse (50%)', color=DOURADO, fontsize=11, ha='left')
ax3.set_ylim(40, 65)
ax3.set_xticks(x)
ax3.set_xticklabels(rot, fontsize=11)
ax3.set_ylabel('Pares em que a pessoa mais alta ganha mais (%)', fontsize=11)
ax3.set_title('Entre duas pessoas iguais em tudo, a mais alta ganha mais pouco acima da metade das vezes\n'
              f'Monte Carlo com {mil(SORTEIOS)} pares sorteados dos resíduos do modelo', fontsize=13, pad=14)
ax3.legend(frameon=False, fontsize=11, loc='upper right')
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-149-grafico-03-pares.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# GRÁFICO 4 · A altura informada contra a medida
fig4, ax4 = plt.subplots(figsize=(12, 7))
for sx, nome, cor in (('1', 'Homens', CINZA), ('2', 'Mulheres', VERMELHO)):
    g = med[med['sexo'] == sx]
    ax4.hist(g['dif'].clip(-15, 15), bins=np.arange(-15.5, 16.5, 1), alpha=0.6, color=cor,
             label=f"{nome}: {vir(g['dif'].mean())} cm a mais, em média")
ax4.axvline(0, color='#333', linewidth=1.5)
ax4.set_xlabel('Altura informada menos altura medida (cm)', fontsize=12)
ax4.set_ylabel('Pessoas', fontsize=12)
ax4.set_title('A gente se lembra um pouco mais alto do que é\n'
              f'{mil(len(med))} pessoas com as duas medidas na PNS 2019', fontsize=13, pad=14)
ax4.legend(frameon=False, fontsize=11)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-149-grafico-04-a-fita-metrica.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")


### 💡 O Insight

**No Brasil, entre pessoas iguais em escola, cor e lugar, cada 10 cm de altura correspondem a cerca de 8% a mais de renda. Entre duas pessoas assim, a mais alta ganha mais em 53% das vezes. Se a altura não importasse, seriam 50%.**

A maior parte da diferença que a gente vê na rua não é a altura. É a infância que veio com ela. O que sobra é pequeno numa pessoa só, mas é real, e o estudo dos genes mostra que pelo menos uma parte é causa. A altura aparece antes de muito do que podemos mostrar e alguém do outro lado da mesa está prestando atenção. Em resumo, a altura influencia pouco o salário de cada pessoa, mas o que ela carrega da infância influencia bastante.

*O que mais entra na sala antes de você, e quanto isso tem pesado nas suas conversas sobre dinheiro?*

---

### ⚠️ Limitações do Modelo

- A altura da camada principal é informada, não medida. Em média, homens se dizem 0,5 cm mais
  altos e mulheres 1 cm mais altas do que a fita mostra. Na subamostra medida o prêmio fica
  parecido, com intervalo largo.
- O rendimento é o do trabalho principal, informado pela própria pessoa, sem renda de aluguel,
  aplicações ou benefícios. Quem não sabe a própria altura (cerca de 16% dos entrevistados) ficou
  fora, e pode ser diferente de quem sabe.
- Regressão não é causa. Mesmo com escola, cor e lugar descontados, sobra o que não foi medido,
  como saúde na infância e renda dos pais. A parte causal vem de outro país, pelo estudo genético
  do Reino Unido.
- Descontar a escolaridade pode tirar demais. Se a altura ajuda a pessoa a estudar mais, parte do
  efeito dela passa pela escola, e os 8% são um piso, não o efeito total.
- A probabilidade dos pares supõe que os resíduos do modelo representam bem a variação de renda
  entre pessoas parecidas. É uma leitura do modelo, não um sorteio de pessoas reais.
- O valor de Judge e Cable é em dólar de 2004 e o de Tyrrell é renda familiar em libras. Eles
  entram como referência, não como número para converter.
- Sem ×0,80, porque são coeficientes, não proporções de pesquisa.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
